# 02 — Expense Classification

Rule-based classification of transactions into Income, Essential Expense,
Discretionary Expense, and Savings. No ML required.

In [0]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import plotly.express as px
from src.data_loader import load_data, preprocess, get_user_data
from src.expense_classifier import (
    classify_all, get_category_breakdown, detect_unusual_expenses, CATEGORY_MAP
)

In [0]:
df = preprocess(load_data('../data/virtual_financial_advisor_data.csv'))
print('Category mapping:')
for cat, cls in sorted(CATEGORY_MAP.items()):
    print(f'  {cat:20s} → {cls}')

## 1. Classify All Transactions

In [0]:
classified = classify_all(df)
print(classified['classification'].value_counts())
classified[['transaction_id', 'category', 'amount', 'classification']].head(10)

## 2. Category Breakdown

In [0]:
breakdown = get_category_breakdown(classified)
breakdown

In [0]:
# Pie chart of expenses by classification
expenses = classified[classified['amount'] < 0].copy()
expenses['abs_amount'] = expenses['amount'].abs()
class_totals = expenses.groupby('classification')['abs_amount'].sum().reset_index()

fig = px.pie(class_totals, names='classification', values='abs_amount',
             title='Expense Distribution by Classification', hole=0.4)
fig.show()

## 3. Unusual Expenses

In [0]:
unusual = detect_unusual_expenses(df, threshold_multiplier=2.0)
print(f'Flagged {len(unusual)} unusual expenses out of {len(df[df["amount"] < 0])} total expenses')
unusual[['date', 'user_id', 'category', 'amount', 'merchant']].head(15)

## 4. Per-User Classification (Sample)

In [0]:
user_df = get_user_data(df, 'user_1')
user_classified = classify_all(user_df)
user_expenses = user_classified[user_classified['amount'] < 0]
user_expenses['abs_amount'] = user_expenses['amount'].abs()

fig2 = px.pie(user_expenses.groupby('classification')['abs_amount'].sum().reset_index(),
              names='classification', values='abs_amount',
              title='user_1 — Expense Classification', hole=0.4)
fig2.show()